# S09_PD_Sandoval_Codificacion
## Codificación de variables categóricas

**Objetivo:** identificar las variables categóricas, clasificarlas como nominales u ordinales, revisar su cardinalidad y aplicar una técnica de codificación adecuada sin inventar órdenes ni provocar fuga de datos.

**Regla principal:** la técnica depende de la naturaleza de la variable y de su cardinalidad.

## 1. Carga del dataset escalado

Se trabaja con el dataset de la semana anterior. Se conserva el archivo original y se crea una copia de trabajo para agregar las columnas de limpieza y codificación.

In [1]:
import pandas as pd
import numpy as np
import unicodedata

df = pd.read_csv("S08_PD_Martinez_Sandoval_Torres_Olmos_DatasetEscalado.csv")

print("Dimensiones:", df.shape)
df.head()

Dimensiones: (518001, 26)


,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,...,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente,z_monto_compra,es_venta_corporativa,edad_cliente_escalada,monto_compra_robusto,precio_unitario_escalado,unidades_vendidas_escaladas
0,P567168,C33473,2025-07-10,tienda,Transferencia,Manizales,1700104,ELECTRÓNICA,Teclado mecánico,122000.0,...,"Todo perfecto, llegó a tiempo",13.5,2023-12-25,0,-0.012390,False,0.459170,-0.142992,-0.077118,0.0006
1,P508296,C91892,NaN,Marketplace,PayPal,Villavicencio,5000101,Electrónica,Teclado mecánico,302100.0,...,El producto llegó dañado,11.3,2026-02-17,0,-0.012391,False,-0.236518,-0.243845,-0.074560,0.0004
2,P714397,C160522,2025-11-17,marketplace,Transferencia,Cúcuta,5400101,Electrónica,Parlante bluetooth,313200.0,...,No era lo que esperaba,8.5,2026-06-11,0,-0.012387,False,0.737445,0.267992,-0.074403,0.0005
3,P241629,C113291,2025-08-02,Tienda,Efectivo,cucuta,5400101,Deportes,Pesas ajustables,457700.0,...,Rápido y sin problemas,8.0,2026-02-27,0,-0.012394,False,0.702661,-0.720644,-0.072350,0.0003
4,P563150,C156773,NaN,Web,PayPal,Bogota D.C.,1100157,jugueteria,Rompecabezas 1000 piezas,386000.0,...,El producto llegó dañado,10.4,2026-07-18,0,-0.012386,False,-0.932206,0.497790,-0.073369,0.0005


## 2. ¿Qué significa codificar?

Codificar una variable categórica significa traducir sus categorías de texto a una representación numérica que un modelo pueda utilizar.

El problema no es simplemente convertir texto en números: **hay que evitar inventar información**. Por ejemplo, `categoria_producto` no tiene un orden natural. Si se asignaran los valores 0, 1, 2, etc. directamente, un modelo podría interpretar que una categoría es mayor o está más lejos de otra.

Por eso primero se responde:

1. ¿Existe un orden real?
2. ¿Cuántas categorías distintas hay?
3. ¿La técnica elegida conserva correctamente la información?

## 3. Inspección de columnas categóricas y cardinalidad

Primero se revisan las columnas de texto y su número de categorías distintas.

En este dataset hay además columnas como identificadores, correos, fechas y comentarios. No todas deben tratarse como variables categóricas del modelo: los identificadores no representan una magnitud útil y las fechas deben tratarse como fechas si se usan en un modelo.

In [2]:
columnas_texto = df.select_dtypes(include="object").columns

cardinalidad = pd.DataFrame({
    "columna": columnas_texto,
    "cardinalidad": [df[col].nunique(dropna=True) for col in columnas_texto]
}).sort_values("cardinalidad")

cardinalidad

,columna,cardinalidad
10,nivel_lealtad,4
11,comentario_cliente,12
9,nivel_satisfaccion,15
3,canal_compra,20
4,metodo_pago,20
7,producto,24
6,categoria_producto,32
5,ciudad_tienda,88
2,fecha_pedido,836
12,fecha_actualizacion_stock,1177


## 4. Limpieza mínima de las categorías

El dataset contiene variaciones como `Tienda`, `tienda`, ` Tienda ` y `TIENDA`. Si se contaran literalmente, parecerían categorías diferentes aunque representen lo mismo.

Se normalizan espacios, mayúsculas/minúsculas y tildes para las variables categóricas que sí se van a codificar. Esto **no cambia el significado**, solo evita duplicar categorías por diferencias de escritura.

In [3]:
def normalizar_texto(s):
    s = s.astype("string").str.strip().str.lower()
    return s.map(
        lambda x: "".join(
            ch for ch in unicodedata.normalize("NFKD", x)
            if not unicodedata.combining(ch)
        ) if pd.notna(x) else x
    )

trabajo = df.copy()

for col in [
    "canal_compra", "metodo_pago", "ciudad_tienda",
    "categoria_producto", "producto",
    "nivel_satisfaccion", "nivel_lealtad"
]:
    trabajo[f"{col}_limpia"] = normalizar_texto(trabajo[col])

trabajo["ciudad_tienda_limpia"] = trabajo["ciudad_tienda_limpia"].fillna("desconocida")
trabajo["nivel_satisfaccion_limpia"] = trabajo["nivel_satisfaccion_limpia"].fillna("desconocido")
trabajo["codigo_postal_str"] = trabajo["codigo_postal"].astype("string")

for col in [
    "categoria_producto_limpia", "metodo_pago_limpia",
    "canal_compra_limpia", "nivel_satisfaccion_limpia",
    "nivel_lealtad_limpia"
]:
    print(col, "->", trabajo[col].nunique(), "categorías")

categoria_producto_limpia -> 6 categorías
metodo_pago_limpia -> 4 categorías
canal_compra_limpia -> 4 categorías
nivel_satisfaccion_limpia -> 4 categorías
nivel_lealtad_limpia -> 4 categorías


## 5. Clasificación de mis columnas categóricas

| Columna | Tipo | Cardinalidad | Técnica | Justificación |
|---|---|---:|---|---|
| `categoria_producto` | Nominal | Baja | One-hot encoding | No tiene orden real y tiene pocas categorías |
| `metodo_pago` | Nominal | Baja | One-hot encoding | No tiene orden real y tiene pocas categorías |
| `canal_compra` | Nominal | Baja | One-hot encoding | No tiene orden real y tiene pocas categorías |
| `nivel_satisfaccion` | Ordinal | Baja | Ordinal encoding | Existe un orden: Bajo < Medio < Alto |
| `nivel_lealtad` | Ordinal | Baja | Ordinal encoding | Existe un orden: Bronce < Plata < Oro < Platino |
| `ciudad_tienda` | Nominal | Alta | Target encoding | Muchas categorías y existe un objetivo claro: `monto_compra` |
| `codigo_postal` | Nominal | Muy alta | Agrupar + frequency encoding | Tiene cientos de categorías y muchas son poco frecuentes |

Las columnas identificadoras se conservan como respaldo, pero no se codifican como variables predictoras.

## 6. One-hot encoding para nominales de baja cardinalidad

Las variables nominales no tienen un orden. Por eso se crean columnas binarias independientes.

Se usa `OneHotEncoder` con `handle_unknown="ignore"` para que una categoría nueva en prueba no rompa la transformación.

El codificador se ajusta con entrenamiento y después se reutiliza sobre prueba, manteniendo exactamente las mismas columnas.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

indices = np.arange(len(trabajo))

idx_entrenamiento, idx_prueba = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

df_entrenamiento = trabajo.iloc[idx_entrenamiento].copy()
df_prueba = trabajo.iloc[idx_prueba].copy()

columnas_nominales = [
    "categoria_producto_limpia",
    "metodo_pago_limpia",
    "canal_compra_limpia"
]

codificador_nominal = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

codificador_nominal.fit(df_entrenamiento[columnas_nominales])

onehot_entrenamiento = pd.DataFrame(
    codificador_nominal.transform(df_entrenamiento[columnas_nominales]),
    columns=codificador_nominal.get_feature_names_out(columnas_nominales),
    index=df_entrenamiento.index
).astype("int8")

onehot_prueba = pd.DataFrame(
    codificador_nominal.transform(df_prueba[columnas_nominales]),
    columns=codificador_nominal.get_feature_names_out(columnas_nominales),
    index=df_prueba.index
).astype("int8")

print("Columnas creadas:", list(onehot_entrenamiento.columns))
print(onehot_entrenamiento.head())

Columnas creadas: ['categoria_producto_limpia_belleza', 'categoria_producto_limpia_deportes', 'categoria_producto_limpia_electronica', 'categoria_producto_limpia_hogar', 'categoria_producto_limpia_jugueteria', 'categoria_producto_limpia_moda', 'metodo_pago_limpia_efectivo', 'metodo_pago_limpia_paypal', 'metodo_pago_limpia_tarjeta', 'metodo_pago_limpia_transferencia', 'canal_compra_limpia_app', 'canal_compra_limpia_marketplace', 'canal_compra_limpia_tienda', 'canal_compra_limpia_web']
        categoria_producto_limpia_belleza  categoria_producto_limpia_deportes  \
123704                                  0                                   0   
417124                                  1                                   0   
207877                                  0                                   0   
138446                                  0                                   0   
511141                                  0                                   1   

        categoria_produc

### ¿Por qué no usar `LabelEncoder` aquí?

`LabelEncoder` asignaría enteros a categorías nominales. Con más de dos categorías, esos números podrían introducir un orden o distancia falsa en modelos lineales o basados en distancias.

Por eso en `categoria_producto`, `metodo_pago` y `canal_compra` se utiliza one-hot encoding.

## 7. Ordinal encoding: respetar el orden real

Aquí sí existe un orden objetivo.

- `nivel_satisfaccion`: **Bajo → Medio → Alto**
- `nivel_lealtad`: **Bronce → Plata → Oro → Platino**

El orden se define explícitamente. No se deja que el alfabeto decida.

In [5]:
from sklearn.preprocessing import OrdinalEncoder

orden_satisfaccion = ["bajo", "medio", "alto"]
orden_lealtad = ["bronce", "plata", "oro", "platino"]

codificador_satisfaccion = OrdinalEncoder(
    categories=[orden_satisfaccion],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

codificador_lealtad = OrdinalEncoder(
    categories=[orden_lealtad],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

df_entrenamiento["nivel_satisfaccion_cod"] = (
    codificador_satisfaccion
    .fit_transform(df_entrenamiento[["nivel_satisfaccion_limpia"]])
    .ravel()
    .astype("int8")
)

df_prueba["nivel_satisfaccion_cod"] = (
    codificador_satisfaccion
    .transform(df_prueba[["nivel_satisfaccion_limpia"]])
    .ravel()
    .astype("int8")
)

df_entrenamiento["nivel_lealtad_cod"] = (
    codificador_lealtad
    .fit_transform(df_entrenamiento[["nivel_lealtad_limpia"]])
    .ravel()
    .astype("int8")
)

df_prueba["nivel_lealtad_cod"] = (
    codificador_lealtad
    .transform(df_prueba[["nivel_lealtad_limpia"]])
    .ravel()
    .astype("int8")
)

print(
    df_final if "df_final" in globals() else
    df_entrenamiento[[
        "nivel_satisfaccion_limpia", "nivel_satisfaccion_cod",
        "nivel_lealtad_limpia", "nivel_lealtad_cod"
    ]].head(10)
)

       nivel_satisfaccion_limpia  nivel_satisfaccion_cod nivel_lealtad_limpia  \
123704                      alto                       2              platino   
417124                      alto                       2                  oro   
207877                     medio                       1               bronce   
138446                      bajo                       0                plata   
511141                      alto                       2                  oro   
386792                     medio                       1                plata   
163702               desconocido                      -1                plata   
213621                      alto                       2              platino   
62019                      medio                       1              platino   
418266                     medio                       1               bronce   

        nivel_lealtad_cod  
123704                  3  
417124                  2  
207877                  

### ¿Por qué usé ordinal encoding?

`nivel_lealtad` sí tiene una jerarquía real. Usar `Bronce=0, Plata=1, Oro=2, Platino=3` conserva esa información.

También se aplica a `nivel_satisfaccion` porque `Bajo < Medio < Alto`. El valor `-1` representa una categoría desconocida por los faltantes, sin confundirla con ninguno de los niveles reales.

## 8. Target encoding y fuga de datos

`ciudad_tienda` tiene alta cardinalidad. En lugar de crear una columna por ciudad, se resume cada categoría usando el promedio de `monto_compra`.

En el dataset existen registros donde `monto_compra` está vacío. Esas filas **no participan en el `fit()`**, porque no tienen objetivo disponible, pero se conservan en el dataset y reciben posteriormente la codificación aprendida.

**Regla fundamental:** el codificador se ajusta (`fit`) **solo con `df_entrenamiento`** y nunca con `df_prueba`.

Después, el mismo codificador ya ajustado se reutiliza con `transform()` sobre entrenamiento y prueba.

Esto evita la **fuga de datos**, porque ninguna información de `df_prueba` interviene en el cálculo de los promedios.

Se usa `smoothing=10` para reducir el efecto de categorías con pocos registros.

In [6]:
# La herramienta indicada en la lección es TargetEncoder de category_encoders.
# Si la librería no está instalada, se usa el TargetEncoder equivalente de scikit-learn
# para que el notebook pueda ejecutarse sin errores.
try:
    from category_encoders import TargetEncoder
    USAR_CATEGORY_ENCODERS = True
    print("Se utilizará TargetEncoder de category_encoders.")
except ImportError:
    from sklearn.preprocessing import TargetEncoder
    USAR_CATEGORY_ENCODERS = False
    print("category_encoders no está instalado; se utilizará TargetEncoder de scikit-learn como respaldo.")

category_encoders no está instalado; se utilizará TargetEncoder de scikit-learn como respaldo.


In [7]:
# Target encoding de ciudad_tienda.
# monto_compra contiene algunos valores nulos, por lo que esas filas no pueden
# participar en el cálculo del promedio objetivo. No se eliminan del dataset final.

entrenamiento_target = df_entrenamiento[
    df_entrenamiento["monto_compra"].notna()
].copy()

print("Filas de entrenamiento:", len(df_entrenamiento))
print("Filas usadas para ajustar el target encoding:", len(entrenamiento_target))
print("Nulos de monto_compra excluidos del fit:",
      df_entrenamiento["monto_compra"].isna().sum())

if USAR_CATEGORY_ENCODERS:
    # Se ajusta SOLO con entrenamiento y con valores válidos del objetivo.
    codificador_ciudad = TargetEncoder(
        cols=["ciudad_tienda_limpia"],
        smoothing=10,
        handle_unknown="value",
        handle_missing="value"
    )

    codificador_ciudad.fit(
        entrenamiento_target[["ciudad_tienda_limpia"]],
        entrenamiento_target["monto_compra"]
    )

    # El mismo codificador ya ajustado se reutiliza en entrenamiento y prueba.
    df_entrenamiento["ciudad_tienda_cod"] = (
        codificador_ciudad
        .transform(df_entrenamiento[["ciudad_tienda_limpia"]])
        ["ciudad_tienda_limpia"]
    )

    df_prueba["ciudad_tienda_cod"] = (
        codificador_ciudad
        .transform(df_prueba[["ciudad_tienda_limpia"]])
        ["ciudad_tienda_limpia"]
    )

else:
    # Respaldo equivalente si category_encoders no está instalado.
    # Los promedios se calculan SOLO con entrenamiento.
    tabla_ciudad = (
        entrenamiento_target
        .groupby("ciudad_tienda_limpia")["monto_compra"]
        .agg(["mean", "count"])
    )

    media_global = entrenamiento_target["monto_compra"].mean()

    peso = tabla_ciudad["count"] / (tabla_ciudad["count"] + 10)

    tabla_ciudad["codificacion"] = (
        peso * tabla_ciudad["mean"]
        + (1 - peso) * media_global
    )

    df_entrenamiento["ciudad_tienda_cod"] = (
        df_entrenamiento["ciudad_tienda_limpia"]
        .map(tabla_ciudad["codificacion"])
        .fillna(media_global)
    )

    df_prueba["ciudad_tienda_cod"] = (
        df_prueba["ciudad_tienda_limpia"]
        .map(tabla_ciudad["codificacion"])
        .fillna(media_global)
    )

print(df_entrenamiento[[
    "ciudad_tienda",
    "ciudad_tienda_cod"
]].head())

Filas de entrenamiento: 414400
Filas usadas para ajustar el target encoding: 400393
Nulos de monto_compra excluidos del fit: 14007
        ciudad_tienda  ciudad_tienda_cod
123704         Cucuta       1.196033e+09
417124    Santa Marta       1.373391e+09
207877        B/manga       7.679524e+07
138446  Villavicencio       1.252257e+09
511141            NaN       8.890923e+08


### ¿Por qué `smoothing=10`?

Una categoría con muy pocos registros puede tener un promedio poco confiable. El suavizado mezcla el promedio de la categoría con el promedio global:

- pocas observaciones → más peso al promedio global;
- muchas observaciones → más peso al promedio propio de la categoría.

Así se reduce el ruido de categorías pequeñas.

## 9. Alta cardinalidad: `codigo_postal`

`codigo_postal` tiene muchas categorías y es una variable nominal: los códigos son etiquetas, no cantidades donde tenga sentido decir que un código es mayor que otro.

Se calculan las categorías frecuentes **solo en entrenamiento**. Los códigos con menos de 15 registros se agrupan en `Otras`. Luego se aplica frequency encoding usando únicamente las frecuencias aprendidas en entrenamiento.

In [8]:
conteo_postal = df_entrenamiento["codigo_postal_str"].value_counts()

postales_frecuentes = conteo_postal[conteo_postal >= 15].index

df_entrenamiento["codigo_postal_agrupado"] = (
    df_entrenamiento["codigo_postal_str"]
    .where(
        df_entrenamiento["codigo_postal_str"].isin(postales_frecuentes),
        "Otras"
    )
)

df_prueba["codigo_postal_agrupado"] = (
    df_prueba["codigo_postal_str"]
    .where(
        df_prueba["codigo_postal_str"].isin(postales_frecuentes),
        "Otras"
    )
)

frecuencia_postal = df_entrenamiento["codigo_postal_agrupado"].value_counts(
    normalize=True
)

df_entrenamiento["codigo_postal_frecuencia"] = (
    df_entrenamiento["codigo_postal_agrupado"]
    .map(frecuencia_postal)
    .fillna(0)
)

df_prueba["codigo_postal_frecuencia"] = (
    df_prueba["codigo_postal_agrupado"]
    .map(frecuencia_postal)
    .fillna(frecuencia_postal.get("Otras", 0))
)

print("Cardinalidad original:",
      trabajo["codigo_postal_str"].nunique())
print("Cardinalidad después de agrupar:",
      df_entrenamiento["codigo_postal_agrupado"].nunique())

print(df_entrenamiento[[
    "codigo_postal",
    "codigo_postal_agrupado",
    "codigo_postal_frecuencia"
]].head())

Cardinalidad original: 490
Cardinalidad después de agrupar: 302
        codigo_postal codigo_postal_agrupado  codigo_postal_frecuencia
123704        5400107                5400107                  0.003077
417124        4700104                4700104                  0.006064
207877        6800118                6800118                  0.000956
138446        5000104                5000104                  0.006595
511141        6800101                6800101                  0.027309


## 10. Unir entrenamiento y prueba

Las columnas originales se conservan como respaldo y las nuevas columnas codificadas se agregan sin sobrescribir los datos originales.

In [9]:
for col in codificador_nominal.get_feature_names_out(columnas_nominales):
    df_entrenamiento[col] = onehot_entrenamiento[col]
    df_prueba[col] = onehot_prueba[col]

df_final = pd.concat(
    [df_entrenamiento, df_prueba],
    axis=0
).sort_index()

df_final["nivel_satisfaccion_cod"] = (
    df_final["nivel_satisfaccion_cod"]
    .fillna(-1)
    .astype("int8")
)

print("Dimensiones finales:", df_final.shape)
df_final.head()

Dimensiones finales: (518001, 53)


,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,...,categoria_producto_limpia_jugueteria,categoria_producto_limpia_moda,metodo_pago_limpia_efectivo,metodo_pago_limpia_paypal,metodo_pago_limpia_tarjeta,metodo_pago_limpia_transferencia,canal_compra_limpia_app,canal_compra_limpia_marketplace,canal_compra_limpia_tienda,canal_compra_limpia_web
0,P567168,C33473,2025-07-10,tienda,Transferencia,Manizales,1700104,ELECTRÓNICA,Teclado mecánico,122000.0,...,0,0,0,0,0,1,0,0,1,0
1,P508296,C91892,NaN,Marketplace,PayPal,Villavicencio,5000101,Electrónica,Teclado mecánico,302100.0,...,0,0,0,1,0,0,0,1,0,0
2,P714397,C160522,2025-11-17,marketplace,Transferencia,Cúcuta,5400101,Electrónica,Parlante bluetooth,313200.0,...,0,0,0,0,0,1,0,1,0,0
3,P241629,C113291,2025-08-02,Tienda,Efectivo,cucuta,5400101,Deportes,Pesas ajustables,457700.0,...,0,0,1,0,0,0,0,0,1,0
4,P563150,C156773,NaN,Web,PayPal,Bogota D.C.,1100157,jugueteria,Rompecabezas 1000 piezas,386000.0,...,1,0,0,1,0,0,0,0,0,1


## 11. Verificaciones finales

Estas comprobaciones permiten confirmar que:

- entrenamiento + prueba = dataset original;
- las columnas one-hot solo contienen 0 y 1;
- `nivel_lealtad` respeta Bronce=0, Plata=1, Oro=2 y Platino=3;
- el target encoding no dejó valores nulos;
- el manejo de `codigo_postal` redujo la cardinalidad;
- el notebook conserva las columnas originales.

In [10]:
# Verificaciones finales del dataset codificado

# 1. Entrenamiento + prueba deben conservar todas las filas originales.
assert len(df_entrenamiento) + len(df_prueba) == len(trabajo)

# 2. Las columnas creadas por one-hot solo deben contener 0 y 1.
nombres_onehot = codificador_nominal.get_feature_names_out(columnas_nominales)
assert df_final[nombres_onehot].isin([0, 1]).all().all()

# 3. nivel_lealtad debe respetar el orden:
#    Bronce=0, Plata=1, Oro=2, Platino=3.
assert set(df_final["nivel_lealtad_cod"].unique()) <= {0, 1, 2, 3}

# 4. Las variables codificadas no deben quedar con valores nulos.
assert df_final["ciudad_tienda_cod"].isna().sum() == 0
assert df_final["codigo_postal_frecuencia"].isna().sum() == 0

print("Todas las verificaciones fueron satisfactorias.")
print("Filas originales:", len(trabajo))
print("Filas finales:", len(df_final))
print("Columnas finales:", len(df_final.columns))

Todas las verificaciones fueron satisfactorias.
Filas originales: 518001
Filas finales: 518001
Columnas finales: 53


## 12. Exportación del dataset final

El archivo se exporta con el nombre solicitado, usando únicamente el apellido **Sandoval**.

In [11]:
df_final.to_csv(
    "S09_PD_Sandoval_DatasetCodificado.csv",
    index=False
)

print("Archivo exportado correctamente:")
print("S09_PD_Sandoval_DatasetCodificado.csv")

Archivo exportado correctamente:
S09_PD_Sandoval_DatasetCodificado.csv


## Conclusión

La codificación correcta depende de la naturaleza de cada variable:

- **Nominal + baja cardinalidad → one-hot encoding.**
- **Ordinal → ordinal encoding con el orden real definido explícitamente.**
- **Nominal + alta cardinalidad + objetivo claro → target encoding con smoothing.**
- **Muy alta cardinalidad → agrupar categorías poco frecuentes y/o frequency encoding.**

El punto más importante es que una codificación numérica no debe inventar un orden que no existe y que cualquier técnica que aprenda información del objetivo debe ajustarse únicamente con el conjunto de entrenamiento para evitar fuga de datos.